In [ ]:
import requests
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


urls = [
    "https://services.swpc.noaa.gov/json/solar_regions.json",
    "https://services.swpc.noaa.gov/json/sunspot_report.json",
    "https://services.swpc.noaa.gov/json/solar_probabilities.json",
    "https://services.swpc.noaa.gov/json/edited_events.json"
]

for url in urls:
    response = requests.get(url)
    print(f"Status: {response.status_code}")

    if response.status_code == 200:
        data = response.json()
        print(f"✅ {url.split('/')[-1]}: {len(data)} événements\n")

        if data:
            print("Premier événement :\n")
            print(json.dumps(data[0], indent=2)[:400])
            print("\n" + "="*50 + "\n")
    else:
        print(f"❌ Erreur: {response.text[:200]}\n")

In [ ]:
# Récupère les données
url = "https://services.swpc.noaa.gov/json/edited_events.json"
response = requests.get(url)
data = response.json()

# Crée un DataFrame
df = pd.DataFrame(data)

# Affiche les infos basiques
print("📊 STRUCTURE DES DONNÉES")
print(f"Nombre de lignes: {len(df)}")
print(f"Nombre de colonnes: {len(df.columns)}\n")

print("Colonnes et types :")
print(df.dtypes)
print("\n")

print("Premières lignes :")
print(df.head())

In [ ]:
print("\n📈 STATISTIQUES DESCRIPTIVES\n")

print("Colonnes avec valeurs null :")
print(df.isnull().sum())
print("\n")

print("Types d'événements uniques :")
print(df['type'].value_counts())
print("\n")

print("Observatoires uniques :")
print(df['observatory'].value_counts())
print("\n")

print("Distribution du quality score :")
print(df['quality'].describe())

In [ ]:
# 1. Distribution des types d'événements
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Type d'événements
df['type'].value_counts().head(10).plot(kind='bar', ax=axes[0,0], color='skyblue')
axes[0,0].set_title('Types d\'événements (top 10)')
axes[0,0].set_xlabel('Type')

# Quality distribution
df['quality_numeric'] = pd.to_numeric(df['quality'], errors='coerce')
df['quality_numeric'].hist(bins=20, ax=axes[0,1], color='coral')
axes[0,1].set_title('Distribution du Quality Score')
axes[0,1].set_xlabel('Quality')

# Observatoires
df['observatory'].value_counts().head(8).plot(kind='barh', ax=axes[1,0], color='lightgreen')
axes[1,0].set_title('Observatoires (top 8)')

# Coded type distribution
df['coded_type'].value_counts().head(8).plot(kind='bar', ax=axes[1,1], color='gold')
axes[1,1].set_title('Coded Type (top 8)')

plt.tight_layout()
plt.show()

print("✅ Graphiques affichés")

In [ ]:
# Convertir en datetime
df['begin_datetime'] = pd.to_datetime(df['begin_datetime'])
df['end_datetime'] = pd.to_datetime(df['end_datetime'])

print("\n⏰ PATTERNS TEMPORELS\n")

print("Période couverte :")
print(f"Du : {df['begin_datetime'].min()}")
print(f"Au : {df['begin_datetime'].max()}\n")

# Événements par jour
df['date'] = df['begin_datetime'].dt.date
events_per_day = df.groupby('date').size()

print(f"Nombre d'événements par jour :")
print(events_per_day.describe())
print(f"\nJour avec le plus d'événements: {events_per_day.idxmax()} ({events_per_day.max()} événements)")
print(f"Jour avec le moins d'événements: {events_per_day.idxmin()} ({events_per_day.min()} événements)")

In [ ]:
print("\n🤖 VARIABLES POTENTIELLES POUR LE ML\n")

# Quelles colonnes sont utilisables ?
print("Colonnes numériques :")
print(df.select_dtypes(include=[np.number]).columns.tolist())

print("\nColonnes texte :")
print(df.select_dtypes(include=['object']).columns.tolist())

print("\nColonnes avec le moins de missing :")
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values()
print(missing_pct[missing_pct < 50])  # Moins de 50% de missing